In [2]:
# Dataset generation
import pandas as pd
import numpy as np

# Simulating a dataset
data = {
    'Age': np.random.randint(20, 60, size=100).astype(float),  # Random ages between 20 and 60
    'State': np.random.choice(['Karnataka', 'Tamil Nadu', 'Maharashtra', 'Delhi', 'Telangana'], size=100),
    'Education': np.random.choice(['High School', 'UG', 'PG'], size=100),
    'Package': np.random.rand(100) * 100  # Random package values for demonstration
}

# Introducing missing values in 'Age' column (5%)
np.random.seed(0)  # For reproducibility
missing_indices = np.random.choice(data['Age'].shape[0], replace=False, size=int(data['Age'].shape[0] * 0.05))
data['Age'][missing_indices] = np.nan

df = pd.DataFrame(data)
df.head()

,Age,State,Education,Package
0,54.0,Tamil Nadu,High School,40.612049
1,49.0,Delhi,PG,56.921076
2,NaN,Telangana,High School,34.360550
3,54.0,Delhi,PG,78.887278
4,33.0,Delhi,PG,41.137241


In [3]:
# Splitting the dataset
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(df.drop(columns=['Package']), df['Package'], test_size=0.2, random_state=42)

In [ ]:
# Encoding using category encoders -> CountEncoder
import sklearn
from sklearn.compose import ColumnTransformer
from category_encoders.count import CountEncoder
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder

preprocessor = ColumnTransformer(
    transformers=[
        ('age_missing', SimpleImputer(strategy='mean'), ['Age']),
        ('cat_state', CountEncoder(normalize=True), ['State']), # parameter - normalize = True is a new transformer called FrequencyEncoder | CountEncoder
        ('education_ordinal', OrdinalEncoder(), ['Education'])
    ]
)

sklearn.set_config(transform_output="pandas")
preprocessor.fit_transform(X_train)

,age_missing__Age,cat_state__State,education_ordinal__Education
55,38.666667,0.2125,0.0
88,42.000000,0.1750,1.0
26,38.666667,0.1750,1.0
42,39.000000,0.2250,1.0
69,33.000000,0.2250,0.0
...,...,...,...
60,45.000000,0.2125,0.0
71,44.000000,0.2125,0.0
14,50.000000,0.1750,2.0
92,31.000000,0.1750,2.0


In [11]:
# Simulating a dataset
np.random.seed(42)  # For reproducibility
data = {
    'State': np.random.choice(['Karnataka', 'Tamil Nadu', 'Maharashtra', 'Delhi', 'Telangana', None], size=100),
    'Education': np.random.choice(['High School', 'UG', 'PG', None], size=100)
}

df = pd.DataFrame(data)
df.isnull().sum()

State        17
Education    23
dtype: int64

In [ ]:
# Parameters
import category_encoders as ce

# Initialize the CountEncoder with various parameters
encoder = ce.CountEncoder(
    cols=['State', 'Education'],  # Specify columns to encode. None would automatically select all categorical columns.
    handle_missing='return_nan',  # Treat NaNs as a countable category. Options are ‘error’, ‘return_nan’, and ‘value’. Default ‘value’,
    handle_unknown='return_nan',  # Treat unknown categories as NaNs (if seen during transform but not in fit). Options are ‘error’ ‘return_nan’, ‘value’ and int.
)

# Fit and transform the dataset
encoder.fit_transform(df).isnull().sum()

State        0
Education    0
dtype: int64

In [49]:
encoder.mapping

{'State': State
 Delhi          25.0
 Tamil Nadu     19.0
 Telangana      17.0
 NaN             NaN
 Maharashtra    11.0
 Karnataka      11.0
 Name: count, dtype: float64,
 'Education': Education
 PG             34.0
 High School    27.0
 NaN             NaN
 UG             16.0
 Name: count, dtype: float64}

In [50]:
new_data = pd.DataFrame({'State': ['Bihar'], 'Education': ['UG']}) # New State
encoder.transform(new_data)

,State,Education
0,NaN,16.0


In [51]:
np.random.seed(0)  # For reproducibility
data = {
    'Category': np.random.choice(['A', 'B', 'C', 'D', 'E', 'F', np.nan], size=100, p=[0.3, 0.25, 0.15, 0.15, 0.05, 0.05, 0.05]),
    'Value': np.random.rand(100)
}

df = pd.DataFrame(data)

df.sample(10)


,Category,Value
91,C,0.209844
29,B,0.290078
2,C,0.735194
50,C,0.149448
44,C,0.806194
78,A,0.704414
33,C,0.298282
65,B,0.855803
75,A,0.223925
45,C,0.703889


In [52]:
df['Category'].value_counts()

,count
Category,
A,34
B,22
C,21
D,12
nan,5
F,4
E,2


In [53]:
encoder = ce.CountEncoder(
    cols=['Category'],
    min_group_size=10,  # Groups with counts less than 5 will be combined
    # min_group_name='salman',  # Use default naming for combined minimum groups
)

# Fit and transform the dataset
encoded_df = encoder.fit_transform(df['Category'])

# Display the original and encoded data for comparison
df['Encoded'] = encoded_df
print(df.head(20))

   Category     Value  Encoded
0         B  0.677817       22
1         D  0.270008       12
2         C  0.735194       21
3         B  0.962189       22
4         B  0.248753       22
5         C  0.576157       21
6         B  0.592042       22
7         E  0.572252       11
8       nan  0.223082       11
9         B  0.952749       22
10        D  0.447125       12
11        B  0.846409       22
12        C  0.699479       21
13        F  0.297437       11
14        A  0.813798       34
15        A  0.396506       34
16        A  0.881103       34
17        D  0.581273       12
18        D  0.881735       12
19        E  0.692532       11


In [54]:
encoder.mapping

{'Category': Category
 A          34
 B          22
 C          21
 D          12
 E_F_nan    11
 Name: count, dtype: int64}

### Binary Encoder

In [55]:
import pandas as pd
import category_encoders as ce

# Sample dataset
data = {
    'Item': ['Item1', 'Item2', 'Item3', 'Item4', 'Item5', 'Item6', 'Item7', 'Item8'],
    'Fruit': ['Apple', 'Banana', 'Cherry', 'Date', 'Elderberry', 'Fig', 'Grape', 'Honeydew']
}
df = pd.DataFrame(data)
df

,Item,Fruit
0,Item1,Apple
1,Item2,Banana
2,Item3,Cherry
3,Item4,Date
4,Item5,Elderberry
5,Item6,Fig
6,Item7,Grape
7,Item8,Honeydew


In [56]:
# Initialize the Binary Encoder
encoder = ce.BinaryEncoder(cols=['Fruit'], return_df=True)

# Fit and transform the data
df_encoded = encoder.fit_transform(df)

# Display the original and encoded data
print(df_encoded)

    Item  Fruit_0  Fruit_1  Fruit_2  Fruit_3
0  Item1        0        0        0        1
1  Item2        0        0        1        0
2  Item3        0        0        1        1
3  Item4        0        1        0        0
4  Item5        0        1        0        1
5  Item6        0        1        1        0
6  Item7        0        1        1        1
7  Item8        1        0        0        0


The problemof using this encoder technique

You cannot use this technique with `interpretation problme` because the columns which was created after transformation has no interperation ability.